# Lab 06-04 · Nodal Analysis: a Mini-SPICE and a Power Grid

Companion to chapter [06-04](../../chapters/06-04-nodal-analysis.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import shutil
import subprocess
import time
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)

ngspice available: True


## Part 1 — A mini-SPICE in 40 lines

The function below reads a netlist in SPICE syntax (resistors `R`, independent current sources `I` and voltage sources `V`, values in plain numbers), assigns a row to every non-ground node and to every voltage source, adds each element's **stamp** (chapter §3.5), and solves the MNA system. SPICE's conventions are kept: `I name n+ n- value` drives current from `n+` through the source to `n-`, and the reported current of a voltage source flows *into* its + terminal.

**🔮 Predict:** for Example 06-04.3 (3 non-ground nodes, 2 voltage sources), what size is the MNA matrix, and how many of its entries are zero?

In [2]:
def mna(netlist):
    '''Solve a netlist of R, I and V elements by modified nodal analysis.
    Returns (node voltages dict, voltage-source currents dict, matrix A, right side z).'''
    elems = []
    for line in netlist.strip().splitlines():
        line = line.split("*")[0].strip()
        if line:
            name, a, b, val = line.split()
            elems.append((name, a, b, float(val)))
    nodes = sorted({n for _, a, b, _ in elems for n in (a, b) if n != "0"})
    vsrcs = [e[0] for e in elems if e[0][0].upper() == "V"]
    row = {n: k for k, n in enumerate(nodes)}
    row.update({s: len(nodes) + k for k, s in enumerate(vsrcs)})
    size = len(nodes) + len(vsrcs)
    A, z = np.zeros((size, size)), np.zeros(size)
    for name, a, b, val in elems:
        kind = name[0].upper()
        pa, pb = row.get(a), row.get(b)          # None for ground
        if kind == "R":
            g = 1.0 / val
            for p, q in ((pa, pb), (pb, pa)):
                if p is not None:
                    A[p, p] += g
                    if q is not None:
                        A[p, q] -= g
        elif kind == "I":                         # current leaves node a, enters node b
            if pa is not None:
                z[pa] -= val
            if pb is not None:
                z[pb] += val
        elif kind == "V":                         # + at a, - at b; unknown j into the + terminal
            m = row[name]
            if pa is not None:
                A[pa, m] += 1; A[m, pa] += 1
            if pb is not None:
                A[pb, m] -= 1; A[m, pb] -= 1
            z[m] = val
    x = np.linalg.solve(A, z)
    v = {n: x[row[n]] for n in nodes}
    j = {s: x[row[s]] for s in vsrcs}
    return v, j, A, z

ex1 = '''
I1 0 1 4
I2 0 2 1
R1 1 0 2
R2 1 2 2
R3 2 0 4
'''
ex3 = '''
V12 1 0 12
V5  2 3 5
R12 1 2 2
R13 1 3 3
R2  2 0 4
R3  3 0 6
'''
p7 = '''
V1 1 0 10
R1 1 2 2
R2 2 0 8
I1 0 2 1
'''
for title, net in (("Example 06-04.1", ex1), ("Examples 06-04.3/4", ex3), ("Problem 06-04.7", p7)):
    v, j, A, z = mna(net)
    print(title, {k: round(float(x), 4) for k, x in v.items()}, {k: round(float(x), 4) for k, x in j.items()})

v, j, A, z = mna(ex3)
print("\nMNA matrix of Example 06-04.3:")
print(np.array2string(A, precision=4, suppress_small=True))
print(f"size {A.shape[0]} x {A.shape[1]}, {np.count_nonzero(A)} nonzeros, {A.size - np.count_nonzero(A)} zeros")
assert A.shape == (5, 5) and A.size - np.count_nonzero(A) == 12
assert np.allclose(A, A.T)                       # symmetric for R and independent sources
assert abs(v["2"] - 10) < 1e-12 and abs(v["3"] - 5) < 1e-12
assert abs(j["V12"] + 10 / 3) < 1e-12 and abs(j["V5"] + 1.5) < 1e-12

Example 06-04.1 {'1': 7.0, '2': 6.0} {}
Examples 06-04.3/4 {'1': 12.0, '2': 10.0, '3': 5.0} {'V12': -3.3333, 'V5': -1.5}
Problem 06-04.7 {'1': 10.0, '2': 9.6} {'V1': -0.2}

MNA matrix of Example 06-04.3:
[[ 0.8333 -0.5    -0.3333  1.      0.    ]
 [-0.5     0.75    0.      0.      1.    ]
 [-0.3333  0.      0.5     0.     -1.    ]
 [ 1.      0.      0.      0.      0.    ]
 [ 0.      1.     -1.      0.      0.    ]]
size 5 x 5, 13 nonzeros, 12 zeros


The matrix is symmetric, and its lower-right 2 × 2 block (the voltage-source rows and columns) is zero. That zero block is why the MNA matrix, unlike the pure nodal matrix, is not positive definite: a simulator factors it with pivoting.

## Part 2 — Agreeing with ngspice

The lab netlist [nodal.cir](nodal.cir) contains the same circuits. Run it and compare every value with the mini-SPICE. (The netlist prints six significant digits, so that is the agreement to look for.)

In [3]:
def run_ngspice(path):
    out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) == 2:
            try:
                vals[parts[0].strip()] = float(parts[1])
            except ValueError:
                pass
    return vals

mini = {}
v1, _, _, _ = mna(ex1); mini["ex1v1"], mini["ex1v2"] = v1["1"], v1["2"]
v3, j3, _, _ = mna(ex3); mini["ex3v2"], mini["ex3v3"], mini["ex4j1"], mini["ex4j2"] = v3["2"], v3["3"], j3["V12"], j3["V5"]
v7, j7, _, _ = mna(p7); mini["p7v2"], mini["p7j"] = v7["2"], j7["V1"]
if HAVE_NGSPICE:
    ng = run_ngspice("nodal.cir")
    for k, x in mini.items():
        print(f"{k:6s} mini-SPICE {x: .9f}   ngspice {ng[k]: .9f}")
        assert abs(x - ng[k]) <= 1e-5 * max(1.0, abs(x))   # ngspice echoes 6 significant digits
else:
    print("ngspice not found; see SETUP.md. Skipping the comparison.")
    for k, x in mini.items():
        print(f"{k:6s} mini-SPICE {x: .9f}")

ex1v1  mini-SPICE  7.000000000   ngspice  7.000000000
ex1v2  mini-SPICE  6.000000000   ngspice  6.000000000
ex3v2  mini-SPICE  10.000000000   ngspice  10.000000000
ex3v3  mini-SPICE  5.000000000   ngspice  5.000000000
ex4j1  mini-SPICE -3.333333333   ngspice -3.333330000
ex4j2  mini-SPICE -1.500000000   ngspice -1.500000000
p7v2   mini-SPICE  9.600000000   ngspice  9.600000000
p7j    mini-SPICE -0.200000000   ngspice -0.200000000


Agreement to every digit ngspice prints is expected: both programs solve the same linear system, and for a linear circuit there is nothing to approximate. (For the diode in `nodal.cir`, ngspice iterates Newton's method with companion models, as in Problem 06-04.13.)

## Part 3 — IR drop in a power grid

Example 06-04.5: a 5 × 5 grid, 10 mΩ per edge, 0.5 A drawn at every node, bumps at 0.80 V on the four corners. A real grid has millions of nodes, so we build the matrix **sparse** from the start (`scipy.sparse`), never as a dense array.

**🔮 Predict:** if a fifth bump is added at the center, how much does the worst drop improve, and where does the worst node move?

In [4]:
def grid_ir(n, r_edge, i_load, vdd, bumps):
    '''Static IR drop of an n x n grid with bumps (list of (row, col)) held at vdd.
    Returns the n x n voltage map and the sparse nodal matrix of the unknown nodes.'''
    idx = lambda a, c: a * n + c
    fixed = {idx(a, c) for a, c in bumps}
    unknown = [k for k in range(n * n) if k not in fixed]
    pos = {k: m for m, k in enumerate(unknown)}
    rows, cols, vals = [], [], []
    b = np.full(len(unknown), -i_load)
    g = 1.0 / r_edge
    for a in range(n):
        for c in range(n):
            for aa, cc in ((a, c + 1), (a + 1, c)):
                if aa < n and cc < n:
                    k, m = idx(a, c), idx(aa, cc)
                    for p, q in ((k, m), (m, k)):
                        if p in pos:
                            rows.append(pos[p]); cols.append(pos[p]); vals.append(g)
                            if q in pos:
                                rows.append(pos[p]); cols.append(pos[q]); vals.append(-g)
                            else:
                                b[pos[p]] += g * vdd
    G = sp.csr_matrix((vals, (rows, cols)), shape=(len(unknown), len(unknown)))
    v = np.full(n * n, vdd)
    v[unknown] = spla.spsolve(G.tocsc(), b)
    return v.reshape(n, n), G

corners = [(0, 0), (0, 4), (4, 0), (4, 4)]
v4, G4 = grid_ir(5, 0.010, 0.5, 0.80, corners)
v5, G5 = grid_ir(5, 0.010, 0.5, 0.80, corners + [(2, 2)])
for label, v, G in (("4 corner bumps", v4, G4), ("+ center bump", v5, G5)):
    worst = np.unravel_index(np.argmin(v), v.shape)
    print(f"{label:15s}: {G.shape[0]} unknowns, {G.nnz} nonzeros, worst {v.min():.6f} V "
          f"(drop {(0.80 - v.min()) * 1e3:.3f} mV) at {tuple(int(x) for x in worst)}")
assert abs(v4.min() - 0.779375) < 1e-9 and G4.nnz == 85
assert abs((0.80 - v5.min()) * 1e3 - 10.0) < 1e-9

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
for ax, v, title in zip(axes, (v4, v5), ("bumps at the corners", "plus a center bump")):
    im = ax.imshow((0.80 - v) * 1e3, cmap="viridis_r", vmin=0, vmax=21)
    ax.set_title(title, fontsize=10); ax.set_xticks(range(5)); ax.set_yticks(range(5)); ax.grid(False)
fig.colorbar(im, ax=axes, label="IR drop (mV)")
plt.show()

4 corner bumps : 21 unknowns, 85 nonzeros, worst 0.779375 V (drop 20.625 mV) at (2, 2)
+ center bump  : 20 unknowns, 76 nonzeros, worst 0.790000 V (drop 10.000 mV) at (4, 2)


The center bump halves the worst drop (20.6 → 10.0 mV), and the worst nodes move to the middles of the edges, now the points farthest from any bump. The corner nodes' and the center's own loads no longer pass through the grid at all.

## Part 4 — Sparse versus dense

Time a dense solve (`numpy.linalg.solve`, which ignores the zeros) and a sparse solve (`scipy.sparse.linalg.spsolve`) for $N \times N$ grids with all four corners bumped.

**🔮 Predict:** dense LU costs about $n^3/3$ operations for $n \approx N^2$ unknowns. If $N$ doubles, by what factor should the dense time grow? And the sparse time?

In [5]:
def timed(f, repeat=3):
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter(); f(); best = min(best, time.perf_counter() - t0)
    return best

results = []
for N in (10, 20, 30, 40, 50, 70, 100, 150):
    _, G = grid_ir(N, 0.010, 0.5, 0.80, [(0, 0), (0, N - 1), (N - 1, 0), (N - 1, N - 1)])
    b = np.ones(G.shape[0])
    Gc = G.tocsc()
    t_sparse = timed(lambda: spla.spsolve(Gc, b))
    t_dense = timed(lambda: np.linalg.solve(G.toarray(), b)) if N <= 50 else float("nan")
    results.append((N, G.shape[0], G.nnz, t_dense, t_sparse))
    print(f"N = {N:4d}: {G.shape[0]:6d} unknowns, {G.nnz:7d} nonzeros "
          f"({100 * G.nnz / G.shape[0] ** 2:.3f}% of dense), dense {t_dense * 1e3:9.2f} ms, sparse {t_sparse * 1e3:8.2f} ms")

R = np.array(results, dtype=float)
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.loglog(R[:, 1], R[:, 3] * 1e3, "o-", label="dense solve")
ax.loglog(R[:, 1], R[:, 4] * 1e3, "s-", label="sparse solve")
ax.set_xlabel("unknowns n"); ax.set_ylabel("time (ms)"); ax.legend(); ax.grid(True, which="both", alpha=0.3)
plt.show()

N =   10:     96 unknowns,     440 nonzeros (4.774% of dense), dense      0.07 ms, sparse     0.14 ms
N =   20:    396 unknowns,    1900 nonzeros (1.212% of dense), dense      2.03 ms, sparse     0.67 ms
N =   30:    896 unknowns,    4360 nonzeros (0.543% of dense), dense     16.87 ms, sparse     1.84 ms


N =   40:   1596 unknowns,    7820 nonzeros (0.307% of dense), dense     36.66 ms, sparse     2.70 ms


N =   50:   2496 unknowns,   12280 nonzeros (0.197% of dense), dense    113.35 ms, sparse     4.48 ms
N =   70:   4896 unknowns,   24200 nonzeros (0.101% of dense), dense       nan ms, sparse     9.61 ms
N =  100:   9996 unknowns,   49580 nonzeros (0.050% of dense), dense       nan ms, sparse    21.96 ms


N =  150:  22496 unknowns,  111880 nonzeros (0.022% of dense), dense       nan ms, sparse    79.62 ms


Your times depend on the machine, so none are asserted. For large $n$ the dense time grows roughly eightfold when $n$ doubles (a factor near 64 when $N$ doubles), while the sparse time grows far more slowly. For the smallest grids the dense solver can win, because its fixed overheads are lower; by $N$ = 50 the sparse solver is far ahead. The dense solver is not even attempted beyond $N$ = 50: at $N$ = 150 its matrix alone would need $(22{,}496)^2 \times 8$ bytes, about 4 GB.

## Your turn

1. Add a 1 kΩ resistor in series with the 5 V source of Example 06-04.3's netlist (you will need a new node). How large is the MNA matrix now, and what are the new node voltages?
2. In Part 3, the grid's metal is made twice as thick (5 mΩ per edge). Predict the new worst drop with four corner bumps, then check.
3. Change Part 4 to bump every node on the grid's edge instead of only the corners. Does the solve get faster or slower, and why?

<details><summary>Answers</summary>

1. One more node (call it x, between the source's − terminal and the 1 kΩ) adds one unknown: a 6 × 6 MNA matrix. With `V5 2 x 5` and `Rs x 3 1000`, `mna()` gives $v_2$ = 8.007 V, $v_3$ = 7.990 V and $v_x$ = 3.007 V, and the 5 V source delivers only 4.98 mA: almost all of its 5 V now appears across the 1 kΩ, so nodes 2 and 3 end up nearly equal, as if joined by a weak link.
2. Every resistance halves and the load currents stay the same, so by linearity every drop halves: 10.3 mV (exactly 20.625/2 = 10.3125 mV).
3. Faster, if only slightly: every bumped node is removed from the unknowns, so the matrix is smaller, and the drop is far smaller because current travels at most half the grid's width.

</details>